In [53]:
import nflreadpy as nfl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import calibration_curve
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, roc_auc_score

In [54]:
DATA_START_SEASON = 2009
MODEL_START_SEASON = 2010
VAL_SEASON = 2024
TEST_SEASON = 2025
SEASONS = list(range(DATA_START_SEASON, TEST_SEASON + 1))


In [55]:
pbp_columns = [
    'game_id',
    'posteam',
    'defteam',
    'epa',
    'success',
    'pass',
    'rush',
    'qb_kneel',
    'qb_spike',
    'two_point_attempt',
    'interception',
    'fumble_lost',
    'sack',
    'qb_dropback',
    'pass_attempt',
    'qb_hit',
    'passer_player_id',
    'passer_player_name',
]

schedule_columns = [
    'game_id',
    'season',
    'week',
    'gameday',
    'game_type',
    'home_team',
    'away_team',
    'home_score',
    'away_score',
    'home_rest',
    'away_rest',
    'location',
    'home_qb_id',
    'away_qb_id',
    'home_qb_name',
    'away_qb_name',
]

In [56]:
pbp = nfl.load_pbp(SEASONS).select(pbp_columns).to_pandas()
schedule = nfl.load_schedules(SEASONS).select(schedule_columns).to_pandas()

In [57]:
TEAM_ABBR_MAP = {
    "SD": "LAC",
    "OAK": "LV",
    "STL": "LA",
}
pbp[["posteam", "defteam"]] = (
    pbp[["posteam", "defteam"]]
    .replace(TEAM_ABBR_MAP)
)
schedule[["home_team", "away_team"]] = (
    schedule[["home_team", "away_team"]]
    .replace(TEAM_ABBR_MAP)
)

In [58]:
scrimmage_plays = pbp.loc[
    pbp['posteam'].notna()
    & pbp['defteam'].notna()
    & pbp['epa'].notna()
    & pbp['success'].notna()
    & (pbp['pass'].eq(1) | pbp['rush'].eq(1))
    & pbp['qb_kneel'].fillna(0).ne(1)
    & pbp['qb_spike'].fillna(0).ne(1)
    & pbp['two_point_attempt'].fillna(0).ne(1)
].copy()

In [59]:
offense = (
    scrimmage_plays
    .groupby(['game_id', 'posteam'])
    .agg(
        off_epa = ('epa', 'mean'),
        off_success_rate = ('success', 'mean')
    )
    .reset_index()
    .rename(columns={'posteam': 'team'})
)

In [60]:
passing = (
    scrimmage_plays.loc[scrimmage_plays["pass"].eq(1)]
    .groupby(['game_id', 'posteam'])
    .agg(pass_epa = ('epa', 'mean'))
    .reset_index()
    .rename(columns={'posteam': 'team'})
)
rush = (
    scrimmage_plays.loc[scrimmage_plays["rush"].eq(1)]
    .groupby(['game_id', 'posteam'])
    .agg(rush_epa = ('epa', 'mean'))
    .reset_index()
    .rename(columns={'posteam': 'team'})
)

In [61]:
defense = (
    scrimmage_plays
    .groupby(['game_id', 'defteam'])
    .agg(
        def_epa_allowed = ('epa', 'mean'),
        def_success_rate_allowed = ('success', 'mean'),
    )
    .reset_index()
    .rename(columns={'defteam': 'team'})
)

In [76]:
home_points = (
    schedule[['game_id', 'home_team', 'home_score', 'away_score']]
    .rename(columns={'home_team': 'team', 'home_score': 'points_scored', 'away_score': 'points_allowed'})
)
away_points = (
    schedule[['game_id', 'away_team', 'home_score', 'away_score']]
    .rename(columns={'away_team': 'team', 'home_score': 'points_allowed', 'away_score': 'points_scored'})
)
points = pd.concat([home_points, away_points], ignore_index=True)
points.sort_values(by=['game_id', 'team'], inplace=True)
points.reset_index(drop=True, inplace=True)
points.head()

,game_id,team,points_scored,points_allowed
0,2009_01_BUF_NE,BUF,24,25
1,2009_01_BUF_NE,NE,25,24
2,2009_01_CHI_GB,CHI,15,21
3,2009_01_CHI_GB,GB,21,15
4,2009_01_DAL_TB,DAL,34,21
